# 第 1 周末练习 —— 技术问答解释器（OpenRouter + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API**（这里走 OpenRouter）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（终端里 `input` 提问）
- **输出**：清晰、结构化的 Markdown 解释
- **额外要求**：用**流式（streaming）**一边生成一边用 `update_display` 刷新显示

这是你在课程期间自己也能天天用的工具：遇到看不懂的概念，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | `system_prompt` 定答法，`get_user_prompt()` 塞问题 |
| 流式输出 `stream=True` | 逐块拼 `answer`，边收边 `update_display` |
| OpenAI 兼容网关 | OpenRouter：`base_url=openrouter_url`，模型名 `openai/gpt-4o-mini` |
| Ollama 本地模型 | OpenAI 兼容口：`http://localhost:11434/v1`，模型 `llama3.2` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. `.env` 里准备 `OPENROUTER_API_KEY`；本地还需 Ollama 已拉取 `llama3.2`
3. 先跑 OpenRouter 那一格，再跑 Ollama 那一格；两格都会再次 `input` 提问，可对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter 的 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：既可调 OpenRouter，也可调 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、display、以及流式刷新用的 update_display
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 环境：加载 .env 并检查 OpenRouter 密钥是否存在 ==========

# override=True：若进程里已有同名环境变量，也用 .env 里的值覆盖（便于笔记本里改密钥后重载）
load_dotenv(override=True)
# 从环境变量读取 OpenRouter API Key（名字必须是 OPENROUTER_API_KEY，和 .env 里一致）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# 有密钥就提示已设置；没有也不抛错，后面调 API 时才会失败——方便先跑通结构
if openrouter_api_key:
    # 可运行英文 print 文案保留：依赖程序/人眼判断的状态字符串不翻译
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")


In [ ]:
# ========== 常量：模型名与两个后端的 base_url 集中写在一处 ==========

# OpenRouter 上的模型路由名：前缀 openai/ 表示走 OpenAI 系模型；字符串必须和网关认识的 id 一致
MODEL_GPT = 'openai/gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'
# OpenRouter 的 OpenAI 兼容 API 根地址（后面传给 OpenAI(base_url=...)）
openrouter_url = "https://openrouter.ai/api/v1"
# 本机 Ollama 的 OpenAI 兼容口（注意是 /v1，不是原生 /api/chat）
ollama_url = "http://localhost:11434/v1"


In [ ]:
# ========== 客户端：用同一套 OpenAI SDK 分别连 OpenRouter 与 Ollama ==========

# OpenRouter 客户端：base_url 指向网关，api_key 用刚才读到的 OPENROUTER_API_KEY
client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
# Ollama 客户端：本地服务通常不校验 key，但 SDK 要求传一个非空字符串，故占位 "ollama"
ollama_client = OpenAI(base_url=ollama_url, api_key="ollama")


In [ ]:
# ========== system prompt：定「怎么答」——发给模型的英文指令保留不译 ==========

# system 角色：全局行为约束；改译会改变回答风格，故正文保持英文
system_prompt = """ 
You are a technical assistant.
Your task is to take a technical question and produce a clear, accurate, and well-structured explanation.
Guidelines:
- Prioritize clarity over complexity.
- Define technical terms briefly when first used.
- Be concise but complete.
- Use structured formatting when helpful:
  - Direct Answer
  - Explanation
  - Example (code or numeric if relevant)
  - Common Pitfalls (if relevant)
- Use bullet points and step-by-step breakdowns for complex topics.
- Do not assume expert-level knowledge unless explicitly stated.
- If the question lacks necessary details, state assumptions clearly.
- Avoid fluff, marketing language, or unnecessary verbosity.
- Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== 构造 user prompt：交互读入问题，再包进发给模型的英文模板 ==========

def get_user_prompt():
    # input：在终端/笔记本里阻塞等待你输入；提示语保持英文（可运行字符串）
    question = input("Please enter your question: ")
    # 回显一遍，方便确认你刚输入的内容
    print(f"Your question: {question}")
    # 把用户问题嵌进英文 user prompt；影响模型行为的字符串不翻译
    user_prompt = f"""
    You are a technical assistant. 
    Please answer the following question in a clear, concise, and structured manner, following the guidelines provided.
    Question: {question}
    """
    # 返回完整 user 文本，供 messages 里 role=user 使用
    return user_prompt


In [ ]:
# ========== 调用 OpenRouter（GPT）：流式生成并边收边刷新 Markdown ==========

# chat.completions.create：发起对话补全；stream=True 表示服务端持续推增量，而不是一次返回全文
stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        # system：答法与格式约束
        {"role": "system", "content": system_prompt},
        # user：现场 input 得到的问题（注意：每跑一次这格都会再问一次）
        {"role": "user", "content": get_user_prompt()}
    ],
    stream=True
)

# display_id=True：拿到可更新的展示句柄，后面用 update_display 原地刷新，而不是刷屏追加
response_display = display(Markdown(""), display_id=True)
# answer：累积已收到的全部文本，用于每次刷新完整 Markdown
answer = ""
# 遍历流式事件：每个 chunk 可能带一小段 delta.content
for chunk in stream:
    # or ""：若本 chunk 没有 content（例如只有 role），就当空串，避免把 None 拼进去
    answer += chunk.choices[0].delta.content or ""
    # 用同一 display_id 更新内容，笔记本里会看到字一个个「长出来」
    update_display(Markdown(answer), display_id=response_display.display_id)


In [ ]:
# ========== 调用本地 Ollama（Llama）：同一套 SDK + 流式刷新，便于对比 ==========

# 与上一格几乎同构，只是客户端换成 ollama_client、模型换成 MODEL_LLAMA
stream_ollama = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        # 再次 get_user_prompt()：会再弹出一次 input，可问同一题或换题对比
        {"role": "user", "content": get_user_prompt()}
    ],
    stream=True
)
# 独立的 display_id，避免和上一格 GPT 的展示互相覆盖
response_display_ollama = display(Markdown(""), display_id=True)
answer_ollama = ""
for chunk in stream_ollama:
    answer_ollama += chunk.choices[0].delta.content or ""
    update_display(Markdown(answer_ollama), display_id=response_display_ollama.display_id)
